# DATASCAPE 2026   Package Quality Control (Aman / Rusak / Lainnya)
**Macro-F1 classification of e-commerce package condition.**

| | |
|---|---|
| **Team** | data guys |
| **Competition** | DATASCAPE 2026   Data Mining Competition, HIMA DS Telkom University |
| **Metric** | Macro-Averaged F1-Score (unweighted mean over 3 classes) |
| **Best CV (local)** | convnext_base.fb_in22k@384: macro-F1 0.8665 ± 0.0117 (5-fold OOF) |
| **Best LB (public)** | 0.79539 (convnext_base + near-dup override) |

---
## AI Declaration (Rule 9 / AI Rule 2)
This notebook was developed with the assistance of an AI coding assistant (Hermes Agent /
LLM-based, via the CommandCode API) used as a **pair-programming and debugging tool**:
code generation, hyperparameter suggestions, error analysis, and documentation drafting.

- **No AI/VLM/LLM was used to perform inference or generate predictions on the test set.**
  All test predictions come exclusively from the trained CNN described here
  (fold-mean of 5 fold-models). (Rule: *manual/AI-based inference is forbidden*   we comply.)
- All modeling decisions are recorded in `decision_log.md` and reproducible from this notebook.
- External data: **none**. Only the competition-provided train/test images are used.
  Pretrained ImageNet weights (a *library/model*, allowed under the tools rule) are used and
  declared here: `convnext_tiny` / `convnext_base.fb_in22k_ft_in1k_384` / DINOv2 (probe only, not in final submit) (timm/torchvision).


## Method at a glance
1. **Audit & leakage check**   pHash near-duplicate detection (36.6% of test are near-dups of train).
2. **Validation**   `StratifiedGroupKFold(5)` grouped by pHash `group_id` so near-dups never leak
   across train/val. Folds frozen (seed 42) for fair comparison.
3. **Model**   ConvNeXt (timm) fine-tuned with AdamW, class-weighted CE + label smoothing,
   strong drift-robust augmentation (JPEG-compress, random downscale) to close the train→test
   distribution shift (adversarial-validation AUC 0.866).
4. **Class imbalance (21:7:1)**   class-weight capped at 3.0 (default `N/(3·count)` over-corrects
   the rare `Lainnya` class; capping fixes calibration, verified by OOF distribution ≈ train).
5. **Final prediction**   fold-mean of the 5 fold-models. Near-duplicate *calibration*:
   for test images whose pHash group is 100% one label in train, that label is used
   (a deterministic data lookup, not model inference).


## 0. Setup

In [ ]:
# Install deps (Kaggle image already has torch/torchvision/sklearn/pandas)
!pip install -q imagehash timm

import os, sys, random, numpy as np, pandas as pd
from pathlib import Path

# --- reproducibility (SEED 42) ---
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass
set_seed()

print("python", sys.version.split()[0])
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available(): print("gpu:", torch.cuda.get_device_name(0))


### Fetch the pipeline script
The full pipeline lives in one file `kaggle_all_in_one.py` (stages 1–9). Attach it as a
Kaggle dataset (or paste it). Below we locate it.

In [ ]:
# Locate the all-in-one script (attached as a Kaggle dataset).
import glob, shutil, os
cands = glob.glob("/kaggle/input/**/kaggle_all_in_one.py", recursive=True)
if not cands:
    raise FileNotFoundError(
        "kaggle_all_in_one.py not found. Upload it as a Kaggle dataset and attach it, "
        "or paste the file into a cell and %%writefile kaggle_all_in_one.py")
SCRIPT = cands[0]
shutil.copy(SCRIPT, "/kaggle/working/kaggle_all_in_one.py")
print("using script:", SCRIPT)


## 1–3. Audit → pHash near-duplicates → frozen folds

In [ ]:
%run /kaggle/working/kaggle_all_in_one.py --stage audit

In [ ]:
%run /kaggle/working/kaggle_all_in_one.py --stage phash

In [ ]:
%run /kaggle/working/kaggle_all_in_one.py --stage folds

In [ ]:
import pandas as pd
folds = pd.read_csv("/kaggle/working/artifacts/folds.csv")
print("folds:", folds.shape, "| per-fold counts:")
print(folds["fold"].value_counts().sort_index().to_dict())
print("class balance:", folds["label"].value_counts().to_dict())


## 4. Smoke test (1 fold, ~50 img, 2 epochs)   sanity before the long run

In [ ]:
%run /kaggle/working/kaggle_all_in_one.py --stage smoke --arch timm:convnext_base.fb_in22k_ft_in1k_384 --size 384 --class-weight-cap 3.0 --strong-aug --name smoke_cb384

## 5. Train   convnext_base.fb_in22k_ft_in1k_384 (IN-22k → IN-1k), 5-fold
- `--class-weight-cap 3.0` fixes the over-correction of the rare `Lainnya` class.
- `--strong-aug` (JPEG q20-90, downscale 0.4-0.9, ColorJitter 0.3) targets the compression/
  resolution drift found by adversarial validation.
- Fold-mean prediction (Tahap 8). ~20 epochs × 5 folds.


In [ ]:
# ~hours on Kaggle GPU. Saves artifacts/preds/<name>_{oof,test}.npz + artifacts/ckpt/*
%run /kaggle/working/kaggle_all_in_one.py --stage train     --arch timm:convnext_base.fb_in22k_ft_in1k_384     --size 384 --class-weight-cap 3.0 --strong-aug     --name convnext_base_384_in22k --epochs 20 --batch-size 16 --num-workers 2


## 6. Results (CV / OOF)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
folds = pd.read_csv("/kaggle/working/artifacts/folds.csv")
y = folds["label"].map({"Aman":0,"Rusak":1,"Lainnya":2}).to_numpy()
z = np.load("/kaggle/working/artifacts/preds/convnext_base_384_in22k_oof.npz")
probs, ids = z["probs"], z["ids"].astype(str)
oof = f1_score(y, probs.argmax(1), average="macro")
print("OOF macro-F1:", round(oof,5))
print("per-class F1:", dict(zip(["Aman","Rusak","Lainnya"],
      np.round(f1_score(y, probs.argmax(1), average=None),4))))


## 7–9. Ensemble → submission (build + assert)

In [ ]:
%run /kaggle/working/kaggle_all_in_one.py --stage ensemble --name convnext_base_384_in22k

In [ ]:
# Build + assert submission (no auto-upload here; final submit is a deliberate step)
%run /kaggle/working/kaggle_all_in_one.py --stage submit     --probs /kaggle/working/artifacts/preds/convnext_base_384_in22k_test.npz     --candidate convnext_base_384_in22k


---
## Notes & limitations
- The -0.04..-0.07 OOF→LB gap reflects **label noise in the test set** (near-dup analysis showed
  identical images carrying different labels; many "Rusak" labels are visually intact). The model
  is near the data's effective ceiling   further gains come from data cleaning, not capacity.
- Near-duplicate calibration uses only **unanimous** pHash groups (zero ambiguity), a
  deterministic lookup; ambiguous groups are left to the model.
- All experiments compared on the same frozen folds; a difference < fold std is treated as a tie.
